# Fake News Detection using Recurrent Neural Networks (RNN)

In this notebook, we'll build a sequential Deep Learning model using an LSTM (Long Short-Term Memory) network to classify news articles as 'Real' or 'Fake'.

## 1. Import Libraries
Import necessary utilities for data manipulation, machine learning metrics, text preprocessing, and TensorFlow functions.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


np.random.seed(42)
tf.random.set_seed(42)

## 2. Load and Prepare the Datasets
Load `Fake.csv` and `True.csv`, label them accordingly, and merge them into a single dataframe.

In [4]:
print("Loading datasets...")
fake_df = pd.read_csv('Fake.csv')
true_df = pd.read_csv('True.csv')


true_df['label'] = 1
fake_df['label'] = 0


data = pd.concat([true_df, fake_df], ignore_index=True)


data['content'] = data['title'] + " " + data['text']


data = data[['content', 'label']]
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total dataset shape: {data.shape}")
data.head()

Loading datasets...
Total dataset shape: (44898, 2)


,content,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,0
1,Failed GOP Candidates Remembered In Hilarious...,0
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,0
3,California AG pledges to defend birth control ...,1
4,AZ RANCHERS Living On US-Mexico Border Destroy...,0


## 3. Text Preprocessing and Train/Test Split
Using Regex to clean out irrelevant components (symbols, URLs) from the articles, and performing our training layout split.

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://[^\s\n\r]+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

print("Cleaning text data...")
data['content'] = data['content'].apply(clean_text)


X = data['content'].values
y = data['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training instances: {len(X_train)} | Testing instances: {len(X_test)}")

Cleaning text data...
Training instances: 35918 | Testing instances: 8980


## 4. Tokenization and Padding
Neural Nets require integers to learn words. We'll tokenize our text and ensure every sequence operates uniformly (zero-padding).

In [6]:
vocab_size = 10000  
max_length = 300     
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)


train_sequences = tokenizer.texts_to_sequences(X_train)
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

test_sequences = tokenizer.texts_to_sequences(X_test)
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

print(f"Sample padded sequence shape: {train_padded[0].shape}")

Sample padded sequence shape: (300,)


## 5. Build the RNN (LSTM) Model
Constructing an architecture combining an Embedding layer (making dense representations of our text inputs) with Recurrence tools (LSTM layer).

In [7]:
embedding_dim = 100

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    LSTM(64), 
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') 
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

c:\Users\chara\anaconda3\envs\ml\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 6. Train the Model
Execute training cycles (epochs) to continuously update validation loss versus input mapping.

In [8]:
num_epochs = 5
batch_size = 128

history = model.fit(
    train_padded, y_train, 
    epochs=num_epochs, 
    validation_data=(test_padded, y_test),
    batch_size=batch_size
)

Epoch 1/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 47s 160ms/step - accuracy: 0.7603 - loss: 0.4870 - val_accuracy: 0.8931 - val_loss: 0.3294
Epoch 2/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 38s 135ms/step - accuracy: 0.7333 - loss: 0.5097 - val_accuracy: 0.5899 - val_loss: 0.6704
Epoch 3/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 37s 131ms/step - accuracy: 0.8082 - loss: 0.3996 - val_accuracy: 0.8795 - val_loss: 0.3554
Epoch 4/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 37s 133ms/step - accuracy: 0.9262 - loss: 0.2375 - val_accuracy: 0.9331 - val_loss: 0.1761
Epoch 5/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 38s 135ms/step - accuracy: 0.9325 - loss: 0.2097 - val_accuracy: 0.9644 - val_loss: 0.1374


## 7. Model Evaluation
Measuring Accuracy, F1-Scores systematically based on predicting across test samples.

In [9]:
model.save("fake_news_model.h5")

In [10]:
predictions = (model.predict(test_padded) > 0.5).astype("int32")

print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['Fake News', 'Real News']))




281/281 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step

Classification Report:
              precision    recall  f1-score   support

   Fake News       0.97      0.96      0.97      4669
   Real News       0.96      0.97      0.96      4311

    accuracy                           0.96      8980
   macro avg       0.96      0.96      0.96      8980
weighted avg       0.96      0.96      0.96      8980



load

In [11]:
from tensorflow.keras.models import load_model

model = load_model("fake_news_model.h5")

## 8. Predict Custom Sentences
Testing random text chunks directly against your trained model to observe outputs in real environments.

In [ ]:
sample_news = [
    "NASA confirms that the Earth is actually flat and they have been hiding the truth for decades, authorities report.", 
    "The Federal Reserve announced a quarter-point interest rate increase on Wednesday as part of its ongoing effort to cool inflation.", 
    "Aliens have landed in Times Square and are demanding to speak with the manager of Earth, causing massive panic.",
    "Iran says peace talks would be 'unreasonable' following Israeli strikes",
    
]


cleaned_samples = [clean_text(text) for text in sample_news]
sample_seq = tokenizer.texts_to_sequences(cleaned_samples)
sample_padded = pad_sequences(sample_seq, maxlen=max_length, padding=padding_type, truncating=trunc_type)

sample_preds = model.predict(sample_padded)
for i, text in enumerate(sample_news):
    label = "Real News" if sample_preds[i][0] > 0.5 else "Fake News"
    confidence = sample_preds[i][0] if label == "Real News" else 1 - sample_preds[i][0]
    print(f"\nNews Content: {text}")
    print(f"Prediction: {label} (Confidence: {confidence:.2%})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step

News Content: NASA confirms that the Earth is actually flat and they have been hiding the truth for decades, authorities report.
Prediction: Fake News (Confidence: 97.62%)

News Content: The Federal Reserve announced a quarter-point interest rate increase on Wednesday as part of its ongoing effort to cool inflation.
Prediction: Real News (Confidence: 99.49%)

News Content: Aliens have landed in Times Square and are demanding to speak with the manager of Earth, causing massive panic.
Prediction: Fake News (Confidence: 97.67%)

News Content: Iran says peace talks would be 'unreasonable' following Israeli strikes
Prediction: Real News (Confidence: 99.52%)

News Content: Iran says peace talks would be 'unreasonable' 
Prediction: Real News (Confidence: 99.36%)
